In [5]:
import pandas as pd
import math
from pathlib import Path

# 1. Paths and Data Loading
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

ELO_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "most_likely_path.txt"

df_elo = pd.read_csv(ELO_PATH)
elo_dict = dict(zip(df_elo['team'], df_elo['current_elo']))

# 2. Math Engine with Reality Check (Wstrzykiwanie wyników z boiska)
known_winners = {
    # Lewa strona drabinki (mecze rozegrane)
    ("Germany", "Paraguay"): "Paraguay",
    ("France", "Sweden"): "France",
    ("South Africa", "Canada"): "Canada",
    ("Netherlands", "Morocco"): "Morocco",
    # Prawa strona drabinki (mecze rozegrane)
    ("Brazil", "Japan"): "Brazil",
    ("Ivory Coast", "Norway"): "Norway",
    ("Mexico", "Ecuador"): "Mexico",
    ("England", "DR Congo"): "England"
}

def predict_match(team_a, team_b):
    # 1. Sprawdzenie, czy mecz już się odbył w rzeczywistości
    if (team_a, team_b) in known_winners:
        winner = known_winners[(team_a, team_b)]
        return winner, 100.0, 100.0, True # True = to fakt historyczny
    elif (team_b, team_a) in known_winners:
        winner = known_winners[(team_b, team_a)]
        return winner, 100.0, 100.0, True

    # 2. Jeśli mecz przed nami -> decyduje matematyka ELO
    elo_a = elo_dict.get(team_a, 1500)
    elo_b = elo_dict.get(team_b, 1500)
    
    prob_a = 1.0 / (1.0 + math.pow(10, (elo_b - elo_a) / 400.0))
    prob_b = 1.0 - prob_a
    
    winner = team_a if prob_a > 0.5 else team_b
    return winner, prob_a * 100, prob_b * 100, False

# 3. FULL Round of 32 Topology
left_bracket_r32 = [
    ("Germany", "Paraguay"),
    ("France", "Sweden"),
    ("South Africa", "Canada"),
    ("Netherlands", "Morocco"),
    ("Portugal", "Croatia"),
    ("Spain", "Austria"),
    ("United States", "Bosnia and Herzegovina"),
    ("Belgium", "Senegal")
]

right_bracket_r32 = [
    ("Brazil", "Japan"),
    ("Ivory Coast", "Norway"),
    ("Mexico", "Ecuador"),
    ("England", "DR Congo"),
    ("Argentina", "Cape Verde"),
    ("Australia", "Egypt"),
    ("Switzerland", "Algeria"),
    ("Colombia", "Ghana")
]

report = []
report.append("🏆 WORLD CUP 2026: FULL 32-TEAM DETERMINISTIC PATH 🏆\n")
report.append("="*65)

def simulate_side(bracket_pairs, side_name):
    report.append(f"\n--- {side_name} BRACKET ---")
    
    # Round of 32
    r16_teams = []
    for team_a, team_b in bracket_pairs:
        winner, p_a, p_b, is_real = predict_match(team_a, team_b)
        if is_real:
            report.append(f"[R32] {team_a} vs {team_b} ---> {winner} (ACTUAL RESULT)")
        else:
            report.append(f"[R32] {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {winner}")
        r16_teams.append(winner)
        
    # Round of 16
    qf_teams = []
    for i in range(0, len(r16_teams), 2):
        team_a, team_b = r16_teams[i], r16_teams[i+1]
        winner, p_a, p_b, _ = predict_match(team_a, team_b)
        report.append(f"[R16] {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {winner}")
        qf_teams.append(winner)

    # Quarter-Finals
    sf_teams = []
    for i in range(0, len(qf_teams), 2):
        team_a, team_b = qf_teams[i], qf_teams[i+1]
        winner, p_a, p_b, _ = predict_match(team_a, team_b)
        report.append(f"[QF]  {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {winner}")
        sf_teams.append(winner)
        
    # Semi-Finals
    team_a, team_b = sf_teams[0], sf_teams[1]
    finalist, p_a, p_b, _ = predict_match(team_a, team_b)
    report.append(f"[SF]  {team_a} ({p_a:.1f}%) vs {team_b} ({p_b:.1f}%) ---> {finalist} GOES TO FINAL")
    
    return finalist

left_finalist = simulate_side(left_bracket_r32, "LEFT")
right_finalist = simulate_side(right_bracket_r32, "RIGHT")

# --- THE GRAND FINAL ---
report.append("\n" + "="*65)
report.append("👑 THE GRAND FINAL 👑")
report.append("="*65)
champion, p_a, p_b, _ = predict_match(left_finalist, right_finalist)
report.append(f"{left_finalist} (Left) ({p_a:.1f}%) vs {right_finalist} (Right) ({p_b:.1f}%)")
report.append(f"\n🏆 PREDICTED WORLD CUP CHAMPION: {champion.upper()} 🏆")

# Printowanie linijka po linijce omija problem z ucinaniem tekstu w VS Code
for line in report:
    print(line)

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(report))

🏆 WORLD CUP 2026: FULL 32-TEAM DETERMINISTIC PATH 🏆


--- LEFT BRACKET ---
[R32] Germany vs Paraguay ---> Paraguay (ACTUAL RESULT)
[R32] France vs Sweden ---> France (ACTUAL RESULT)
[R32] South Africa vs Canada ---> Canada (ACTUAL RESULT)
[R32] Netherlands vs Morocco ---> Morocco (ACTUAL RESULT)
[R32] Portugal (59.1%) vs Croatia (40.9%) ---> Portugal
[R32] Spain (76.3%) vs Austria (23.7%) ---> Spain
[R32] United States (74.4%) vs Bosnia and Herzegovina (25.6%) ---> United States
[R32] Belgium (61.2%) vs Senegal (38.8%) ---> Belgium
[R16] Paraguay (21.1%) vs France (78.9%) ---> France
[R16] Canada (31.0%) vs Morocco (69.0%) ---> Morocco
[R16] Portugal (36.4%) vs Spain (63.6%) ---> Spain
[R16] United States (44.8%) vs Belgium (55.2%) ---> Belgium
[QF]  France (64.6%) vs Morocco (35.4%) ---> France
[QF]  Spain (73.8%) vs Belgium (26.2%) ---> Spain
[SF]  France (49.9%) vs Spain (50.1%) ---> Spain GOES TO FINAL

--- RIGHT BRACKET ---
[R32] Brazil vs Japan ---> Brazil (ACTUAL RESULT)
[R32] I